# 图像分块、对比学习与可训练重建基线

浏览器 Python 实验：按顺序运行，变量在单元之间共享。图表来自当前代码计算。


## 1 · 像素与可逆分块

用合成灰度图把维度压到能手算。修改 patch，检查整除关系。

In [ ]:
# The blog provides live charts; standalone Python prints chart data.
if 'display_plot' not in globals():
    def display_plot(x, y, title='', xlabel='', ylabel=''):
        print(title, list(zip(x,y)))

import math, random
side, patch = 8, 2
assert side % patch == 0
pixels = [[(x+y)/(2*(side-1)) for x in range(side)] for y in range(side)]
patches = [[pixels[y+dy][x+dx] for dy in range(patch) for dx in range(patch)] for y in range(0,side,patch) for x in range(0,side,patch)]
print("image shape:",(side,side),"patch tensor:",(len(patches),patch*patch))
rebuilt = [[0.0]*side for _ in range(side)]
for i,p in enumerate(patches):
    y,x = (i//(side//patch))*patch,(i%(side//patch))*patch
    for j,value in enumerate(p):
        rebuilt[y+j//patch][x+j%patch] = value
assert rebuilt == pixels
print("patchify -> unpatchify: exact round trip")

## 2 · 对比学习的二维玩具例子

手工向量演示归一化与图文双向 loss，不是 CLIP 的预训练输出。把配对次序交换，观察 loss 增大。

In [ ]:
images, texts = [[1,0],[0,1]], [[0.9,0.1],[0.2,0.8]]
def unit(v):
    return [x/sum(z*z for z in v)**0.5 for x in v]
images,texts = [unit(v) for v in images],[unit(v) for v in texts]
temperature = 0.1
S = [[sum(x*y for x,y in zip(a,b))/temperature for b in texts] for a in images]
def ce(row,target):
    m=max(row)
    return math.log(sum(math.exp(x-m) for x in row))+m-row[target]
loss = (sum(ce(S[i],i) for i in range(2))+sum(ce([S[j][i] for j in range(2)],i) for i in range(2)))/4
print("scaled similarity:",S)
print("symmetric contrastive loss:",loss)

## 3 · 隐藏像素，训练重建器

这里训练 z=a*x+b*y+c，只看可见像素，评估隐藏像素。它是可解释的重建基线，不是 MAE；MAE 的完整权重实验保留在 PyTorch Notebook。

In [ ]:
random.seed(19)
mask_ratio = 0.75
assert 0 < mask_ratio < 1
coords = [(x/(side-1),y/(side-1),pixels[y][x]) for y in range(side) for x in range(side)]
masked = set(random.sample(range(side*side),int(side*side*mask_ratio)))
train = [c for i,c in enumerate(coords) if i not in masked]
w = [0.0,0.0,0.0]
losses=[]
for epoch in range(250):
    grad=[0.0]*3
    for x,y,z in train:
        err = w[0]*x+w[1]*y+w[2]-z
        for j,f in enumerate([x,y,1]):
            grad[j] += 2*err*f/len(train)
    w = [a-0.2*g for a,g in zip(w,grad)]
    mse = sum((w[0]*coords[i][0]+w[1]*coords[i][1]+w[2]-coords[i][2])**2 for i in masked)/len(masked)
    losses.append(mse)
print("visible / masked:",len(train),len(masked))
print("learned coefficients:",w)
print("masked MSE:",losses[-1])
display_plot(list(range(250)),losses,"Held-out pixel reconstruction","epoch","masked MSE")

## 4 · 分布变化会怎样

把平滑渐变换成棋盘格，线性函数不能表达纹理；这展示低 loss 的结论依赖数据分布。

In [ ]:
checker = [(x+y)%2 for y in range(side) for x in range(side)]
prediction = [w[0]*x+w[1]*y+w[2] for x,y,z in coords]
print("gradient-image MSE:",sum((p-c[2])**2 for p,c in zip(prediction,coords))/len(coords))
print("checkerboard MSE:",sum((p-z)**2 for p,z in zip(prediction,checker))/len(coords))
display_plot(list(range(side)),prediction[:side],"Predicted first image row","column","intensity")